# Day 12 — Phase diagram of agent failure

**The headline experiment.** Sweep distractor density × gold-passage position. Plot accuracy as a 2D heatmap per model. Look for a phase boundary.

Prereqs:
- working `eval_runner` from notebook 04
- working RAG pipeline + distractor injection from `src/reliability_maps/distractors.py`
- a 100–200 question multi-hop set (HotpotQA recommended)
- run from Day 11 should already be cached, so this notebook re-loads results from disk and only re-runs missing cells

In [ ]:
import sys; sys.path.append("../src")
import asyncio, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from reliability_maps.distractors import generate_perturbation_set, BaseQuestion
from reliability_maps.runners import Runner, EvalCell
from reliability_maps.metrics import accuracy, bootstrap_ci

# TODO: load your prepared questions
QUESTIONS: list[BaseQuestion] = []  # populate from HotpotQA loader
MODELS = ["gpt-4o-mini", "claude-haiku-4-5", "qwen2-0.5b"]

In [ ]:
# Control parameter grids
DENSITY_GRID  = [0, 1, 2, 3, 5, 7, 10, 15]
POSITION_GRID = ["start", "middle", "end"]

axes = {"density": DENSITY_GRID, "position": POSITION_GRID}

# Build the cell list
cells: list[EvalCell] = []
for q in QUESTIONS:
    for prompt, kind, params in generate_perturbation_set(q, axes):
        for model in MODELS:
            provider = "openai" if "gpt" in model else ("anthropic" if "claude" in model else "local")
            cells.append(EvalCell(
                model=model, provider=provider,
                question_id=q.qid,
                perturbation_kind=kind, perturbation_params=params,
                prompt=prompt, gold_answer=q.gold_answer,
            ))
print("total cells:", len(cells))

In [ ]:
runner = Runner(cache_dir=".cache/phase_diagram", max_concurrency=8)
# results = await runner.run(cells, provider_fn=..., scorer=...)
# df = runner.to_dataframe(results)
# df.to_parquet("results/phase_diagram.parquet")

df = pd.read_parquet("results/phase_diagram.parquet")
df.head()

## Headline figure — accuracy heatmap (density × position) per model

In [ ]:
fig, axes_ = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4), sharey=True)
for ax, model in zip(axes_, MODELS):
    sub = df[df["model"] == model]
    pivot = sub.pivot_table(index="position", columns="n_distractors",
                            values="is_correct", aggfunc="mean")
    pivot = pivot.reindex(index=POSITION_GRID, columns=DENSITY_GRID)
    sns.heatmap(pivot, ax=ax, vmin=0, vmax=1, cmap="RdYlGn",
                annot=True, fmt=".2f", cbar=(model == MODELS[-1]))
    ax.set_title(model)
    ax.set_xlabel("# distractor passages")
    ax.set_ylabel("gold position" if model == MODELS[0] else "")
fig.suptitle("Accuracy as a function of distractor density × gold position", y=1.02)
plt.tight_layout()
plt.savefig("figures/phase_diagram_density_position.pdf", bbox_inches="tight")
plt.savefig("figures/phase_diagram_density_position.png", dpi=200, bbox_inches="tight")
plt.show()

## Cross-model overlay — accuracy vs distractor count (gold position fixed to middle)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for model in MODELS:
    sub = df[(df["model"] == model) & (df["position"] == "middle")]
    grouped = sub.groupby("n_distractors")["is_correct"].agg(["mean", "count"])
    # bootstrap CI
    cis = sub.groupby("n_distractors")["is_correct"].apply(
        lambda x: bootstrap_ci(x.tolist())
    )
    means = grouped["mean"]
    lows  = [c[0] for c in cis]
    highs = [c[1] for c in cis]
    ax.plot(means.index, means.values, marker="o", label=model)
    ax.fill_between(means.index, lows, highs, alpha=0.15)
ax.set_xlabel("# distractor passages")
ax.set_ylabel("accuracy")
ax.set_title("Cross-model accuracy degradation (gold passage in middle)")
ax.legend()
plt.savefig("figures/cross_model_overlay.pdf", bbox_inches="tight")
plt.show()

## Looking for a phase transition (informal)

If a model has a sharp drop at some critical $n^*$, the curve looks sigmoidal rather than linear. Useful diagnostics:

1. Fit a logistic to each model's curve. Report the inflection point.
2. Rescale x-axis by the per-model inflection point. Do the curves *collapse* onto a single universal shape? If yes, that's a clean blog hook.
3. Look for **hysteresis**: take a poisoned context, remove the distractor on a follow-up turn — does the agent recover? (This is its own experiment; flag for Day 13.)

Be honest in writeup: this is empirical pattern-matching with physics vocabulary, not a proof of phase-transition dynamics.